# 03 - Clean and Process Data

## Objective

Build the reusable analysis tables from the audited raw data.

This notebook:
- standardizes timestamps and basic data types;
- defines the primary observation window;
- converts document events into document-level clearance outcomes;
- builds a one-row-per-captain analysis table;
- joins approval and activation outcomes;
- derives onboarding and activation timing fields; and
- validates and saves the processed dataset.

The raw CSV files are read only; they are not modified.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

## 1. Load the raw datasets

The seven source tables cover the captain lifecycle, document verification, activation, nudges, and airport operations.

In [2]:
captains = pd.read_csv("../data/captains.csv")
doc_events = pd.read_csv("../data/doc_events.csv")
approvals = pd.read_csv("../data/approvals.csv")
activation = pd.read_csv("../data/activation.csv")
nudges = pd.read_csv("../data/nudges.csv")
airport_hourly = pd.read_csv("../data/airport_hourly.csv")
airport_trips = pd.read_csv("../data/airport_trips.csv")

print("Raw datasets loaded.")

Raw datasets loaded.


In [3]:
dataset_shapes = pd.DataFrame({
    "dataset": [
        "captains", "doc_events", "approvals", "activation",
        "nudges", "airport_hourly", "airport_trips"
    ],
    "rows": [
        len(captains), len(doc_events), len(approvals), len(activation),
        len(nudges), len(airport_hourly), len(airport_trips)
    ],
    "columns": [
        len(captains.columns), len(doc_events.columns), len(approvals.columns),
        len(activation.columns), len(nudges.columns),
        len(airport_hourly.columns), len(airport_trips.columns)
    ]
})

display(dataset_shapes)

,dataset,rows,columns
0,captains,25000,9
1,doc_events,186282,7
2,approvals,25000,5
3,activation,4206,5
4,nudges,16314,6
5,airport_hourly,10248,9
6,airport_trips,60000,9


## 2. Standardize timestamps

All event timestamps are converted to pandas datetime values so that elapsed-time calculations and cutoff checks are consistent.

In [4]:
tables = {
    "captains": captains,
    "doc_events": doc_events,
    "approvals": approvals,
    "activation": activation,
    "nudges": nudges,
    "airport_hourly": airport_hourly,
    "airport_trips": airport_trips,
}

In [5]:
timestamp_columns = {
    "captains": ["signup_ts"],
    "doc_events": ["event_ts"],
    "approvals": ["decision_ts"],
    "activation": ["first_order_ts"],
    "nudges": ["sent_ts"],
    "airport_hourly": ["hour_ts"],
    "airport_trips": ["request_ts"],
}

In [6]:
for table_name, columns in timestamp_columns.items():
    table = tables[table_name]

    for column in columns:
        table[column] = pd.to_datetime(
            table[column],
            errors="coerce"
        )

In [7]:
timestamp_check = pd.DataFrame([
    {
        "dataset": table_name,
        "column": column,
        "dtype": str(tables[table_name][column].dtype),
        "missing": int(tables[table_name][column].isna().sum())
    }
    for table_name, columns in timestamp_columns.items()
    for column in columns
])

display(timestamp_check)

,dataset,column,dtype,missing
0,captains,signup_ts,datetime64[ns],0
1,doc_events,event_ts,datetime64[ns],0
2,approvals,decision_ts,datetime64[ns],20359
3,activation,first_order_ts,datetime64[ns],102
4,nudges,sent_ts,datetime64[ns],0
5,airport_hourly,hour_ts,datetime64[ns],0
6,airport_trips,request_ts,datetime64[ns],0


## 3. Define the observation window

The source data runs through **June 30, 2026**. Recent signups may not have had enough time to complete onboarding by the extraction date.

For the primary onboarding funnel, this notebook flags captains with at least **14 days of observable time**. This is a cohort eligibility flag, not a deletion rule: all captains remain in `captain_base`.

In [8]:
DATA_CUTOFF = pd.Timestamp("2026-06-30 23:59:59")
MIN_OBSERVATION_DAYS = 14

In [9]:
captains["days_observed"] = (
    DATA_CUTOFF - captains["signup_ts"]
).dt.total_seconds() / (24 * 60 * 60)

In [10]:
captains["eligible_for_funnel"] = (
    captains["days_observed"] >= MIN_OBSERVATION_DAYS
)

In [11]:
observation_summary = pd.Series({
    "total_captains": len(captains),
    "funnel_eligible": int(captains["eligible_for_funnel"].sum()),
    "right_censored": int((~captains["eligible_for_funnel"]).sum()),
})

display(observation_summary.to_frame("count"))

,count
total_captains,25000
funnel_eligible,22757
right_censored,2243


In [12]:
captains["days_observed"].describe()

count    25000.000000
mean        85.546792
std         52.144111
min          0.085405
25%         39.534190
50%         83.342002
75%        130.130891
max        180.744433
Name: days_observed, dtype: float64

## 4. Process document events

`doc_events` contains multiple events for each captain-document combination.

For downstream funnel analysis, the event history is reduced to one row per **captain × document type**. A document is marked as cleared if it has at least one `verification_pass` event.

`attempts` uses the maximum recorded `attempt_no`, while the event flags preserve whether an upload, pass, or failure occurred at any point.

In [13]:
doc_summary = (
    doc_events
    .groupby(["captain_id", "doc_type"], as_index=False)
    .agg(
        attempts=("attempt_no", "max"),
        upload_success=("event_type", lambda x: (x == "upload_success").any()),
        verification_pass=("event_type", lambda x: (x == "verification_pass").any()),
        verification_fail=("event_type", lambda x: (x == "verification_fail").any()),
    )
)

In [14]:
first_pass = (
    doc_events.loc[
        doc_events["event_type"].eq("verification_pass"),
        ["captain_id", "doc_type", "event_ts"]
    ]
    .groupby(["captain_id", "doc_type"], as_index=False)["event_ts"]
    .min()
    .rename(columns={"event_ts": "first_pass_ts"})
)

In [15]:
doc_summary = doc_summary.merge(
    first_pass,
    on=["captain_id", "doc_type"],
    how="left",
    validate="one_to_one",
)

In [16]:
display(doc_summary.head())
print("Document-level rows:", len(doc_summary))

,captain_id,doc_type,attempts,upload_success,verification_pass,verification_fail,first_pass_ts
0,CPT100000,AADHAAR,1,True,True,False,2026-05-04 12:16:11.389574146
1,CPT100000,DL,1,True,True,False,2026-05-02 22:37:15.678999608
2,CPT100000,FITNESS,2,True,False,True,NaT
3,CPT100000,PERMIT,1,True,True,False,2026-05-05 14:52:12.185964306
4,CPT100000,RC,1,True,True,False,2026-05-03 17:48:59.486740931


Document-level rows: 81173


In [17]:
doc_summary["verification_fail_without_pass"] = (
    doc_summary["verification_fail"] & ~doc_summary["verification_pass"]
)

In [18]:
display(
    doc_summary["doc_type"].value_counts().rename_axis("doc_type").to_frame("captain_document_rows")
)

,captain_document_rows
doc_type,
DL,23303
RC,19248
AADHAAR,14665
FITNESS,9248
PERMIT,9213
INSURANCE,5496


In [19]:
print(
    "Captain-document records with a failure but no eventual pass:",
    int(doc_summary["verification_fail_without_pass"].sum())
)

Captain-document records with a failure but no eventual pass: 7729


## 5. Create captain-level document features

The funnel is evaluated at captain level, so document outcomes are pivoted into clearance indicators.

Missing document rows are treated as **not cleared** in these boolean features. This does not mean the document failed; it means there is no recorded successful verification event for that captain-document pair.

In [20]:
doc_cleared = (
    doc_summary
    .pivot(
        index="captain_id",
        columns="doc_type",
        values="verification_pass"
    )
    .reset_index()
)

In [21]:
doc_cleared.columns.name = None

doc_cleared = doc_cleared.rename(columns={
    "DL": "DL_cleared",
    "RC": "RC_cleared",
    "AADHAAR": "AADHAAR_cleared",
    "PERMIT": "PERMIT_cleared",
    "FITNESS": "FITNESS_cleared",
    "INSURANCE": "INSURANCE_cleared",
})

In [22]:
doc_columns = [
    "DL_cleared",
    "RC_cleared",
    "AADHAAR_cleared",
    "PERMIT_cleared",
    "FITNESS_cleared",
    "INSURANCE_cleared",
]

In [23]:
for column in doc_columns:
    doc_cleared[column] = (
        doc_cleared[column]
        .astype("boolean")
        .fillna(False)
    )

display(doc_cleared.head())

,captain_id,AADHAAR_cleared,DL_cleared,FITNESS_cleared,INSURANCE_cleared,PERMIT_cleared,RC_cleared
0,CPT100000,True,True,False,False,True,True
1,CPT100001,False,True,False,False,False,False
2,CPT100002,True,True,True,True,True,True
3,CPT100003,False,True,False,False,False,False
4,CPT100004,True,True,True,False,False,True


## 6. Build the captain-level analysis table

`captains` is the signup-level universe, so it remains the base table throughout the joins.

This prevents captains with no document events, no approval decision, or no activation record from disappearing from the analysis.

In [24]:
captain_base = captains.merge(
    doc_cleared,
    on="captain_id",
    how="left",
    validate="one_to_one",
)

In [25]:
for column in doc_columns:
    captain_base[column] = (
        captain_base[column]
        .astype("boolean")
        .fillna(False)
    )

print("Rows:", len(captain_base))
print("Unique captain IDs:", captain_base["captain_id"].nunique())

Rows: 25000
Unique captain IDs: 25000


In [26]:
display(
    captain_base[doc_columns]
    .sum()
    .rename("captains_with_passed_document")
    .to_frame()
)

,captains_with_passed_document
DL_cleared,21954
RC_cleared,15852
AADHAAR_cleared,14095
PERMIT_cleared,8468
FITNESS_cleared,8241
INSURANCE_cleared,4664


## 7. Add approval outcomes

The approval table contains the final onboarding outcome and the last stage reached.

It is joined one-to-one on `captain_id`, while retaining the full signup population.

In [27]:
approval_columns = [
    "captain_id",
    "decision_ts",
    "final_status",
    "last_stage_reached",
    "docs_cleared",
]

In [28]:
captain_base = captain_base.merge(
    approvals[approval_columns],
    on="captain_id",
    how="left",
    validate="one_to_one",
)

In [29]:
print("Rows after approval join:", len(captain_base))
print("Unique captain IDs:", captain_base["captain_id"].nunique())

Rows after approval join: 25000
Unique captain IDs: 25000


In [30]:
display(
    captain_base["final_status"]
    .value_counts(dropna=False)
    .rename_axis("final_status")
    .to_frame("captains")
)

,captains
final_status,
dropped_in_docs,19062
approved,4206
in_progress,1297
rejected,435


## 8. Add activation outcomes

`activation` contains post-approval activity for approved captains.

A left join keeps non-approved and non-activated captains in the analysis table. A missing `first_order_ts` therefore means there is no recorded first order in the activation data; it is not converted to zero.

In [31]:
activation_columns = [
    "captain_id",
    "first_order_ts",
    "orders_d7",
    "orders_d30",
    "online_hours_d30",
]

In [32]:
captain_base = captain_base.merge(
    activation[activation_columns],
    on="captain_id",
    how="left",
    validate="one_to_one",
)

In [33]:
print("Rows after activation join:", len(captain_base))
print("Unique captain IDs:", captain_base["captain_id"].nunique())

Rows after activation join: 25000
Unique captain IDs: 25000


In [34]:
activation_summary = pd.Series({
    "approved captains": int(captain_base["final_status"].eq("approved").sum()),
    "with first order": int(captain_base["first_order_ts"].notna().sum()),
    "without first order": int(captain_base["first_order_ts"].isna().sum()),
})

In [35]:
display(activation_summary.to_frame("count"))

,count
approved captains,4206
with first order,4104
without first order,20896


## 9. Derive lifecycle durations

Two reusable timing fields are created:

- `signup_to_approval_days`: signup → approval decision
- `approval_to_first_order_days`: approval decision → first completed order

Missing timestamps naturally produce missing durations.

In [36]:
captain_base["signup_to_approval_days"] = (
    captain_base["decision_ts"] - captain_base["signup_ts"]
).dt.total_seconds() / (24 * 60 * 60)

In [37]:
captain_base["approval_to_first_order_days"] = (
    captain_base["first_order_ts"] - captain_base["decision_ts"]
).dt.total_seconds() / (24 * 60 * 60)

In [38]:
duration_summary = captain_base[
    ["signup_to_approval_days", "approval_to_first_order_days"]
].describe()

In [39]:
display(duration_summary)

,signup_to_approval_days,approval_to_first_order_days
count,4641.000000,4104.000000
mean,6.429790,1.714744
std,1.547602,1.365636
min,2.542293,0.001303
25%,5.341806,0.721890
50%,6.313038,1.374499
75%,7.387370,2.364522
max,14.109059,11.068693


In [40]:
duration_checks = pd.Series({
    "negative signup-to-approval durations": int(
        (captain_base["signup_to_approval_days"] < 0).sum()
    ),
    "negative approval-to-first-order durations": int(
        (captain_base["approval_to_first_order_days"] < 0).sum()
    ),
})

In [41]:
display(duration_checks.to_frame("count"))

,count
negative signup-to-approval durations,0
negative approval-to-first-order durations,0


## 10. Final integrity checks

Before saving, verify that the processed table still has the intended one-row-per-captain grain and that the key lifecycle joins did not create duplicate rows.

In [42]:
final_checks = pd.Series({
    "rows": len(captain_base),
    "unique captain IDs": captain_base["captain_id"].nunique(),
    "duplicate captain IDs": int(captain_base["captain_id"].duplicated().sum()),
    "negative signup-to-approval": int(
        (captain_base["signup_to_approval_days"] < 0).sum()
    ),
    "negative approval-to-first-order": int(
        (captain_base["approval_to_first_order_days"] < 0).sum()
    ),
})

In [43]:
display(final_checks.to_frame("value"))

,value
rows,25000
unique captain IDs,25000
duplicate captain IDs,0
negative signup-to-approval,0
negative approval-to-first-order,0


In [44]:
missing_summary = (
    captain_base.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing")
    .to_frame()
)

In [45]:
display(missing_summary[missing_summary["missing"] > 0].head(15))

,missing
online_hours_d30,21628
orders_d30,21628
orders_d7,20981
approval_to_first_order_days,20896
first_order_ts,20896
signup_to_approval_days,20359
decision_ts,20359
last_stage_reached,4393


## 11. Save the processed captain dataset

The processed captain-level table is saved as an intermediate dataset for the downstream funnel, campaign, and activation analyses.

In [46]:
OUTPUT_PATH = "../data/captain_base.csv"
captain_base.to_csv(OUTPUT_PATH, index=False)

In [47]:
print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {captain_base.shape}")

Saved: ../data/captain_base.csv
Shape: (25000, 27)


## Conclusion

The raw captain lifecycle data is now represented in a reusable **one-row-per-captain** table.

The processed table:
- keeps the full signup population;
- includes document clearance indicators;
- carries approval outcomes and onboarding stage information;
- includes activation outcomes where available; and
- provides reusable lifecycle duration fields.

Recent signups remain in the table but are flagged through `eligible_for_funnel`, allowing downstream funnel analysis to apply the 14-day observation rule without discarding the underlying records.

The resulting `captain_base.csv` is the main captain-level input for the next analysis notebooks.